In [49]:
library(ggplot2)
library(microbiome)
library(phyloseq)
library("tidyverse")
library("dplyr")
library(vegan)
library(qiime2R)
library(microViz)
library(ComplexUpset)
set.seed(1)
library(SCRuB)

## Change fish species to standard names across databases

### Files run through worms to identify differences

In [2]:
# fish species to update from MIRe

## (python taxassign.py -i /Users/stephanie.rosales/Documents/omics_projects/MIR/metadata/speciesOfIntrest_1col.csv
#-a WoRMS -o /Users/stephanie.rosales/Documents/omics_projects/MIR/metadata/worms_SOI.cvs --n-proc 2)
tax_ref <- tibble::tribble(
  ~verbatimIdentification,              ~scientificName,
  "Acanthostracion polygonius",           "Acanthostracion polygonium",
  "Bairdiella sanctaeluciae",             "Corvula sanctaeluciae",
  "Chaetodon aculeatus",                 "Prognathodes aculeatus",
  "Chromis enchrysura",                  "Chromis enchrysurus",
  "Coryphopterus bol",                   "Coryphopterus venezuelae",
  "Enneanectes pectoralis",               "Enneanectes jordani",
  "Equetus lanceolatus",                 "Eques lanceolatus",
  "Eucinostomus lefroyi",                "Ulaema lefroyi",
  "Gobiosoma macrodon",                  "Tigrigobius macrodon",
  "Haemulon plumieri",                   "Haemulon plumierii",
  "Harengula humeralis",                 "Harengula jaguana",
  "Hemiemblemaria simulus",              "Hemiemblemaria simula",
  "Holocentrus coruscus",                "Sargocentron coruscum",
  "Hypoplectrus sp.",                    "Hypoplectrus",
  "Hypoplectrus Hybrid",                 "Hypoplectrus",
  "Labrisomus gobio",                    "Gobioclinus gobio",
  "Lipogramma trilineatum",              "Lipogramma trilineata",
  "Micrognathus ensenadae",              "Micrognathus crinitus",
  "Opistognathus whitehurstii",           "Opistognathus whitehursti",
  "Paradiplogrammus bairdi",              "Callionymus bairdi",
  "Ptereleotris calliurus",               "Ptereleotris calliura",
  "Sargocentron vexillarium",             "Neoniphon vexillarium",
  "Stephanolepis hispidus",               "Stephanolepis hispida",
  "Tigrigobius saucra",                   "Tigrigobius saucrus",
  "Trachinotus falcatus",                "Trachinotus blochii",
  "Chromis cyanea",                      "Azurina cyanea",
  "Chromis multilineata",                "Azurina multilineata",
  "Clepticus parrae",                    "Brama parrae",
  "Haemulon chrysargyreum",              "Brachygenys chrysargyrea"
)

##(python taxassign.py -i /Users/stephanie.rosales/Documents/omics_projects/MIR/metadata/NCRMP24_1col.csv 
#-a WoRMS -o /Users/stephanie.rosales/Documents/omics_projects/MIR/metadata/worms_ncrmp_fish24.cvs --n-proc 2)

worm_ncrmp= read.csv("/Users/stephanie.rosales/Documents/omics_projects/MIR/metadata/worms_ncrmp_fish24.csv",
              header = TRUE)

dim(worm_ncrmp)
dim(distinct(worm_ncrmp))
head(worm_ncrmp)

[1] 478  15

[1] 478  15

,verbatimIdentification,scientificName,scientificNameID,taxonRank,kingdom,phylum,class,order,family,genus,species,nameAccordingTo,match_type_debug,cleanedTaxonomy,ambiguous
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>,<lgl>
1,Ablennes hians,Ablennes hians,urn:lsid:marinespecies.org:taxname:159246,Species,Animalia,Chordata,Teleostei,Beloniformes,Belonidae,Ablennes,NA,WoRMS,Success_Batch_Ablennes hians,Ablennes hians,FALSE
2,Abudefduf saxatilis,Abudefduf saxatilis,urn:lsid:marinespecies.org:taxname:159288,Species,Animalia,Chordata,Teleostei,Ovalentaria incertae sedis,Pomacentridae,Abudefduf,NA,WoRMS,Success_Batch_Abudefduf saxatilis,Abudefduf saxatilis,FALSE
3,Abudefduf taurus,Abudefduf taurus,urn:lsid:marinespecies.org:taxname:273701,Species,Animalia,Chordata,Teleostei,Ovalentaria incertae sedis,Pomacentridae,Abudefduf,NA,WoRMS,Success_Batch_Abudefduf taurus,Abudefduf taurus,FALSE
4,Acanthemblemaria aspera,Acanthemblemaria aspera,urn:lsid:marinespecies.org:taxname:279443,Species,Animalia,Chordata,Teleostei,Blenniiformes,Chaenopsidae,Acanthemblemaria,NA,WoRMS,Success_Batch_Acanthemblemaria aspera,Acanthemblemaria aspera,FALSE
5,Acanthemblemaria chaplini,Acanthemblemaria chaplini,urn:lsid:marinespecies.org:taxname:279448,Species,Animalia,Chordata,Teleostei,Blenniiformes,Chaenopsidae,Acanthemblemaria,NA,WoRMS,Success_Batch_Acanthemblemaria chaplini,Acanthemblemaria chaplini,FALSE
6,Acanthemblemaria maria,Acanthemblemaria maria,urn:lsid:marinespecies.org:taxname:279457,Species,Animalia,Chordata,Teleostei,Blenniiformes,Chaenopsidae,Acanthemblemaria,NA,WoRMS,Success_Batch_Acanthemblemaria maria,Acanthemblemaria maria,FALSE


In [35]:
write.csv(density, "/Users/stephanie.rosales/Documents/omics_projects/MIR/Fish_comparison/Working_files/ncrmp_fish_24_denisty_worms.csv")


### Original files merged with worms output

In [80]:
# ----Species of Interest (SOI)----

# Read in the list of species of interest
soi_ori <- read.csv("/Users/stephanie.rosales/Documents/omics_projects/MIR/metadata/speciesOfIntrest.csv", header = TRUE)

head(soi_ori )

soi = soi_ori %>%

  # Join to the taxonomic reference table to standardize names
  # Match the SOI scientificName to the verbatimIdentification in tax_ref
  left_join(
    tax_ref,
    by = c("scientificName" = "verbatimIdentification")
  ) %>%
  
  # Preserve the original species name and replace it with the
  # updated/accepted scientific name when available
  mutate(
    scientificName_original = scientificName,
    scientificName = coalesce(scientificName.y, scientificName)
  ) %>%
  
  # Remove the extra joined column to clean up the table
  select(-scientificName.y)

# Inspect the resulting SOI table
head(soi)
dim(soi)

print("Identify species names that were updated via the taxonomic reference for SOI")
setdiff(soi$scientificName, soi$scientificName_original)

print("Count how many species names were changed")
length(setdiff(soi$scientificName, soi$scientificName_original))

write.csv(soi, "/Users/stephanie.rosales/Documents/omics_projects/MIR/Fish_comparison/Working_files/SOI_worm.csv")

# ---- NCRMP Density Data ----
# Read in NCRMP fish density data
remove_ncrmp=c("no observed fishes", "unknown species", "no observed fishes")

ncrmp_density <- read.table(
  "/Users/stephanie.rosales/Documents/omics_projects/MIR/metadata/ncrmp_fish_24_denisty.csv",
  header = TRUE,
  sep = "\t"
) 
head(ncrmp_density)

 density = ncrmp_density %>%
  
  # Flag records as originating from the NCRMP dataset
  mutate(NCRMP = "TRUE") %>%
  
  # Rename GROUP to Species for consistency across datasets
  rename(Species = GROUP) %>%
  
  # Retain only species identity and NCRMP presence
  #select(Species, NCRMP) %>%

  filter(!Species %in% remove_ncrmp) %>%
  # Join to the taxonomic reference table to standardize species names
  left_join(
    #tax_ref,
      worm_ncrmp, 
    by = c("Species" = "verbatimIdentification")
  ) %>%
  
  # Preserve the original species name and replace it with the standardized scientific name when available
  mutate(
    Species_original = Species,
    Species = coalesce(scientificName, Species)
  ) %>%
  
# Remove redundant taxonomic column 
  select(-scientificName)

# Inspect the resulting density table
head(density)
dim(density)

print("Identify species updated via the taxonomic reference")
setdiff(density$Species, density$Species_original)

print("Count how many species names were changed")
length(setdiff(density$Species, density$Species_original))

write.csv(density, "/Users/stephanie.rosales/Documents/omics_projects/MIR/Fish_comparison/Working_files/ncrmp_fish_24_denisty_worms.csv")



,Genus,Species,scientificName
,<chr>,<chr>,<chr>
1,Ablennes,hians,Ablennes hians
2,Abudefduf,saxatilis,Abudefduf saxatilis
3,Abudefduf,taurus,Abudefduf taurus
4,Acanthemblemaria,aspera,Acanthemblemaria aspera
5,Acanthemblemaria,spinosa,Acanthemblemaria spinosa
6,Acanthemblemaria,chaplini,Acanthemblemaria chaplini


,Genus,Species,scientificName,scientificName_original
,<chr>,<chr>,<chr>,<chr>
1,Ablennes,hians,Ablennes hians,Ablennes hians
2,Abudefduf,saxatilis,Abudefduf saxatilis,Abudefduf saxatilis
3,Abudefduf,taurus,Abudefduf taurus,Abudefduf taurus
4,Acanthemblemaria,aspera,Acanthemblemaria aspera,Acanthemblemaria aspera
5,Acanthemblemaria,spinosa,Acanthemblemaria spinosa,Acanthemblemaria spinosa
6,Acanthemblemaria,chaplini,Acanthemblemaria chaplini,Acanthemblemaria chaplini


[1] 373   4

[1] "Identify species names that were updated via the taxonomic reference for SOI"


[1] "Acanthostracion polygonium" "Corvula sanctaeluciae"     
 [3] "Prognathodes aculeatus"     "Azurina cyanea"            
 [5] "Azurina multilineata"       "Chromis enchrysurus"       
 [7] "Brama parrae"               "Coryphopterus venezuelae"  
 [9] "Eques lanceolatus"          "Ulaema lefroyi"            
[11] "Tigrigobius macrodon"       "Haemulon plumierii"        
[13] "Brachygenys chrysargyrea"   "Hemiemblemaria simula"     
[15] "Sargocentron coruscum"      "Hypoplectrus"              
[17] "Gobioclinus gobio"          "Lipogramma trilineata"     
[19] "Micrognathus crinitus"      "Opistognathus whitehursti" 
[21] "Callionymus bairdi"         "Ptereleotris calliura"     
[23] "Neoniphon vexillarium"      "Stephanolepis hispida"     
[25] "Tigrigobius saucrus"        "Trachinotus blochii"

...

[1] 212  26

[1] "Identify species updated via the taxonomic reference"


[1] "Brachygenys chrysargyrea"   "Coryphopterus"             
 [3] "Haemulon"                   "Acanthostracion polygonium"
 [5] "Caranx bartholomaei"        "Azurina cyanea"            
 [7] "Azurina multilineata"       "Brama parrae"              
 [9] "incertae sedis"             "Neoniphon vexillarium"     
[11] "Trachinotus blochii"

[1] "Count how many species names were changed"


[1] 11

## Prepare data to remove sample contamination with SCRUB

In [53]:
#read metadata file and set factor order
meta_m3 = read.csv("/Users/stephanie.rosales/Documents/omics_projects/MIR/metadata//FAIRe-NOAA_noaa-aoml-mire - sampleMetadata.csv",
                skip = 2, header = TRUE, row.names=1) %>%
  filter(!is.na(libID_Marver3), libID_Marver3 != "")
head(meta_m3 , n=2)
dim(meta_m3 )

meta_m3  <- meta_m3[, colSums(is.na(meta_m3)) < nrow(meta_m3)]

rownames(meta_m3) <- meta_m3$libID_Marver3

head(meta_m3, n=2)
dim(meta_m3)


,samp_name_archive,month,year,date,serial_number,extract_id,sample_number,libID_Marver1,libID_Marver3,libID_Mifish,⋯,silicate_unit,tot_alkalinity,tot_alkalinity_unit,transmittance,transmittance_unit,biosample_accession,organism,samp_collect_notes,dna_yield,dna_yield_unit
,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<chr>,<lgl>,<chr>,<chr>
MIRe_10A_Horseshoe_CB-1_2024-05-23,MIR_eDNA1_24_144_c_HS_CB_1_10A,May,2024,5/23/24,HS_CB-1_10A,10A,10,MIRe1_12S_MV1_10A,MIRe1_16S_MV3_10A,MIRe1_12S_MF_10A,⋯,NA,NA,NA,NA,NA,NA,seawater,NA,11.02,ng
MIRe_10B_Horseshoe_CB-1_2024-05-23,MIR_eDNA1_24_144_c_HS_CB_1_10B,May,2024,5/23/24,HS_CB-1_10B,10B,10,MIRe1_12S_MV1_10B,MIRe1_16S_MV3_10B,MIRe1_12S_MF_10B,⋯,NA,NA,NA,NA,NA,NA,seawater,NA,5.2,ng


[1]  95 175

,samp_name_archive,month,year,date,serial_number,extract_id,sample_number,libID_Marver1,libID_Marver3,libID_Mifish,⋯,filter_name,prepped_samp_store_sol,samp_vol_we_dna_ext,samp_vol_we_dna_ext_unit,tot_depth_water_col,temp,tidal_stage,organism,dna_yield,dna_yield_unit
,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
MIRe1_16S_MV3_10A,MIR_eDNA1_24_144_c_HS_CB_1_10A,May,2024,5/23/24,HS_CB-1_10A,10A,10,MIRe1_12S_MV1_10A,MIRe1_16S_MV3_10A,MIRe1_12S_MF_10A,⋯,Sterivex,lysis buffer,2,L,10,29.6,high,seawater,11.02,ng
MIRe1_16S_MV3_10B,MIR_eDNA1_24_144_c_HS_CB_1_10B,May,2024,5/23/24,HS_CB-1_10B,10B,10,MIRe1_12S_MV1_10B,MIRe1_16S_MV3_10B,MIRe1_12S_MF_10B,⋯,Sterivex,lysis buffer,2,L,10,29.6,high,seawater,5.2,ng


[1] 95 64

In [54]:
#read metadata file and set factor order
meta_m1 = read.csv("/Users/stephanie.rosales/Documents/omics_projects/MIR/metadata//FAIRe-NOAA_noaa-aoml-mire - sampleMetadata.csv",
                skip = 2, header = TRUE, row.names=1) %>%
  filter(!is.na(libID_Marver1), libID_Marver1 != "")%>%   mutate(across(where(is.character), ~ str_replace_all(., "MIR-eDNA1", "2024:May"))) %>%
  mutate(across(where(is.character), ~ str_replace_all(., "MIR-eDNA2", "2024:Sept"))) %>%
  mutate(across(where(is.character), ~ str_replace_all(., "MIR-eDNA3", "2025:Feb")))

head(meta_m1, n=2)
dim(meta_m1)

meta_m1$libID_Marver1 <- gsub("MIRe1_12S_", "", meta_m1$libID_Marver1)
meta_m1$libID_Marver1 <- gsub("MIRe2_12S_", "", meta_m1$libID_Marver1)
meta_m1$libID_Marver1 <- gsub("MIRe3_12S_", "", meta_m1$libID_Marver1)

meta_m1$libID_Marver1 <- gsub("MV1_CFN_n_2A", "MV1_CFN", meta_m1$libID_Marver1)
meta_m1$libID_Marver1 <- gsub("MV1_CFS_n_2A", "MV1_CFS", meta_m1$libID_Marver1)
meta_m1$libID_Marver1 <- gsub("MV3_CR_n_3A", "MV3_CR", meta_m1$libID_Marver1)
meta_m1$libID_Marver3 <- gsub("MV3_HS_n_3A", "MV3_HS", meta_m1$libID_Marver1)
meta_m1 <- meta_m1[, colSums(is.na(meta_m1)) < nrow(meta_m1)]

rownames(meta_m1) <- meta_m1$libID_Marver1

head(meta_m1, n=2)
dim(meta_m1)

,samp_name_archive,month,year,date,serial_number,extract_id,sample_number,libID_Marver1,libID_Marver3,libID_Mifish,⋯,silicate_unit,tot_alkalinity,tot_alkalinity_unit,transmittance,transmittance_unit,biosample_accession,organism,samp_collect_notes,dna_yield,dna_yield_unit
,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<chr>,<lgl>,<chr>,<chr>
MIRe_10A_Horseshoe_CB-1_2024-05-23,MIR_eDNA1_24_144_c_HS_CB_1_10A,May,2024,5/23/24,HS_CB-1_10A,10A,10,MIRe1_12S_MV1_10A,MIRe1_16S_MV3_10A,MIRe1_12S_MF_10A,⋯,NA,NA,NA,NA,NA,NA,seawater,NA,11.02,ng
MIRe_10B_Horseshoe_CB-1_2024-05-23,MIR_eDNA1_24_144_c_HS_CB_1_10B,May,2024,5/23/24,HS_CB-1_10B,10B,10,MIRe1_12S_MV1_10B,MIRe1_16S_MV3_10B,MIRe1_12S_MF_10B,⋯,NA,NA,NA,NA,NA,NA,seawater,NA,5.2,ng


[1]  94 175

,samp_name_archive,month,year,date,serial_number,extract_id,sample_number,libID_Marver1,libID_Marver3,libID_Mifish,⋯,filter_name,prepped_samp_store_sol,samp_vol_we_dna_ext,samp_vol_we_dna_ext_unit,tot_depth_water_col,temp,tidal_stage,organism,dna_yield,dna_yield_unit
,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
MV1_10A,MIR_eDNA1_24_144_c_HS_CB_1_10A,May,2024,5/23/24,HS_CB-1_10A,10A,10,MV1_10A,MV1_10A,MIRe1_12S_MF_10A,⋯,Sterivex,lysis buffer,2,L,10,29.6,high,seawater,11.02,ng
MV1_10B,MIR_eDNA1_24_144_c_HS_CB_1_10B,May,2024,5/23/24,HS_CB-1_10B,10B,10,MV1_10B,MV1_10B,MIRe1_12S_MF_10B,⋯,Sterivex,lysis buffer,2,L,10,29.6,high,seawater,5.2,ng


[1] 94 64

In [55]:
#read metadata file and set factor order
meta_mi = read.csv("/Users/stephanie.rosales/Documents/omics_projects/MIR/metadata//FAIRe-NOAA_noaa-aoml-mire - sampleMetadata.csv",
                skip = 2, header = TRUE, row.names=1) %>% 
  filter(!is.na(libID_Mifish), libID_Mifish != "") %>% 
  filter(libID_Mifish!="MIRe1_12S_MF_16B") %>% # remove no seqs
  filter(libID_Mifish!="MIRe2_12S_MF_33B") %>% # remove no seqs
  filter(libID_Mifish!="MIRe2_12S_MF_NTC") %>% # remove no seqs
mutate(across(where(is.character), ~ str_replace_all(., "MIR-eDNA1", "2024:May"))) %>%
  mutate(across(where(is.character), ~ str_replace_all(., "MIR-eDNA2", "2024:Sept"))) %>%
  mutate(across(where(is.character), ~ str_replace_all(., "MIR-eDNA3", "2025:Feb")))  %>%
  mutate(across(where(is.character), ~ str_replace_all(., "Checca_Rocks", "Checca Rocks")))

#meta <- meta[, colSums(is.na(meta)) < nrow(meta)]
rownames(meta_mi) <- meta_mi$libID_Mifish

head(meta_mi, n=2) 
dim(meta_mi)

,samp_name_archive,month,year,date,serial_number,extract_id,sample_number,libID_Marver1,libID_Marver3,libID_Mifish,⋯,silicate_unit,tot_alkalinity,tot_alkalinity_unit,transmittance,transmittance_unit,biosample_accession,organism,samp_collect_notes,dna_yield,dna_yield_unit
,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<chr>,<lgl>,<chr>,<chr>
MIRe1_12S_MF_10A,MIR_eDNA1_24_144_c_HS_CB_1_10A,May,2024,5/23/24,HS_CB-1_10A,10A,10,MIRe1_12S_MV1_10A,MIRe1_16S_MV3_10A,MIRe1_12S_MF_10A,⋯,NA,NA,NA,NA,NA,NA,seawater,NA,11.02,ng
MIRe1_12S_MF_10B,MIR_eDNA1_24_144_c_HS_CB_1_10B,May,2024,5/23/24,HS_CB-1_10B,10B,10,MIRe1_12S_MV1_10B,MIRe1_16S_MV3_10B,MIRe1_12S_MF_10B,⋯,NA,NA,NA,NA,NA,NA,seawater,NA,5.2,ng


[1] 161 175

In [56]:
#read ASV table

ASV_qza_3 <- read_qza("/Users/stephanie.rosales/Documents/omics_projects/MIR/analysis/seq_run2/marver3/qiime_output/table_Marver3_seq2.qza")
ASV_table_3 <- ASV_qza_3$data
head(ASV_table_3, n=2)
dim(ASV_table_3)

#read ASV table
ASV_qza_1 <- read_qza("/Users/stephanie.rosales/Documents/omics_projects/MIR/analysis/seq_run1//marver1/qiime_output//table_Marver1_seq1.qza")
ASV_table_1 <- ASV_qza_1$data
head(ASV_table_1, n=2)
dim(ASV_table_1)

#read ASV table
ASV_qza_mi <- read_qza("/Users/stephanie.rosales/Documents/omics_projects/MIR/analysis/seq_run1/mifish/qiime_output_sfl/table_Mifish_seq1_sfl.qza")
ASV_table_mi <- ASV_qza_mi$data
head(ASV_table_mi, n=2)
dim(ASV_table_mi)

Warning message in readLines(file, warn = readLines.warn):
“incomplete final line found on '/var/folders/j4/j37z3nw57wl5490t5t1rhk7h4k4czw/T//RtmpegPLI4/f3a13815-e359-48a3-bd84-a8fb292f7c66/metadata.yaml'”


,MIRe1_16S_MV3_10A,MIRe1_16S_MV3_10B,MIRe1_16S_MV3_11A,MIRe1_16S_MV3_11B,MIRe1_16S_MV3_12A,MIRe1_16S_MV3_12B,MIRe1_16S_MV3_13A,MIRe1_16S_MV3_13B,MIRe1_16S_MV3_14A,MIRe1_16S_MV3_14B,⋯,MIRe2_16S_MV3_47B,MIRe2_16S_MV3_48A,MIRe2_16S_MV3_48B,MIRe2_16S_MV3_CFN,MIRe2_16S_MV3_CFS,MIRe2_16S_MV3_CR,MIRe2_16S_MV3_ExBlank,MIRe2_16S_MV3_HS,MIRe2_16S_MV3_MSU,MIRe2_16S_MV3_NTC
24603dde52fe88f5148c3c60c5d6e1b2,79820,5257,4819,14456,78936,499167,177978,111711,58426,309238,⋯,52481,46816,115286,347039,0,14894,141,10143,0,0
ece06f5929d636c6850f5444574486f5,0,0,3692,922,15980,578,3359,16764,27704,47524,⋯,15,6,10508,0,0,14,0,0,0,0


[1] 3646   96

Warning message in readLines(file, warn = readLines.warn):
“incomplete final line found on '/var/folders/j4/j37z3nw57wl5490t5t1rhk7h4k4czw/T//RtmpegPLI4/0a1e94ca-571d-445c-aa1c-e95417698541/metadata.yaml'”


,MV1_10A,MV1_10B,MV1_11A,MV1_11B,MV1_12A,MV1_12B,MV1_13A,MV1_13B,MV1_14A,MV1_14B,⋯,MV1_9B,MV1_CFN,MV1_CFS,MV1_CR,MV1_ExBlank,MV1_HS,MV1_MSU,MV1_neg1A,MV1_neg1B,MV1_NTC
e03b6ee0df42c3ca44ea26ed6c4d27aa,105,38519,16102,36958,54713,272060,215056,49484,70268,103651,⋯,37515,9776,137621,87712,0,255833,100942,213134,356518,6
7da05e198a2c205b708d4f3ebeb5ee12,0,9978,0,27847,18627,1068,44412,25778,36252,83416,⋯,7870,1231,0,0,45127,3118,0,0,0,0


[1] 2685   96

Warning message in readLines(file, warn = readLines.warn):
“incomplete final line found on '/var/folders/j4/j37z3nw57wl5490t5t1rhk7h4k4czw/T//RtmpegPLI4/5150b7d2-9611-43a5-8a7e-a3d8e8449c01/metadata.yaml'”


,MIRe1_12S_MF_10A,MIRe1_12S_MF_10B,MIRe1_12S_MF_11A,MIRe1_12S_MF_11B,MIRe1_12S_MF_12A,MIRe1_12S_MF_12B,MIRe1_12S_MF_13A,MIRe1_12S_MF_13B,MIRe1_12S_MF_14A,MIRe1_12S_MF_14B,⋯,MIRe3_12S_MF_72A,MIRe3_12S_MF_72B,MIRe3_12S_MF_CFN_n_3A,MIRe3_12S_MF_CFS_n_3A,MIRe3_12S_MF_CR_n_3A,MIRe3_12S_MF_CR_n_3B,MIRe3_12S_MF_ExBlank,MIRe3_12S_MF_HS_n_3A,MIRe3_12S_MF_NTC,MIRe3_12S_MF_NTC2
5e704746baef0d3e632a7def74b449b9,0,42501,0,0,11,4,14376,7424,8864,0,⋯,9871,0,0,0,503586,0,0,0,0,0
addd57bd3b3937b61b32dece26f8f166,25957,0,0,31805,16427,3,30234,33756,41302,178084,⋯,95071,44794,0,0,0,0,0,0,0,0


[1] 5729  165

In [57]:
taxa_qza <- read_qza("/Users/stephanie.rosales/Documents/omics_projects//MIR//analysis/seq_run2//marver3/qiime_output//taxonomy_marver3_seq2.qza")
taxa_table_3 <- taxa_qza$data %>%
as_tibble() %>% 
#filter(Taxon!="Unassigned") %>%
#select(-Consensus) %>% 
separate(Taxon, sep=";", c("Domain", "Phylum", "Class", "Order", "Family", "Genus", "Species")) %>%
arrange(Feature.ID) %>%
mutate(ASVs =paste('ASV',1:n(), sep = "_")) %>%
#mutate(Genus = sub("g__", "", Genus)) %>%
#mutate(Genus = sub("_[0-9]+", "", Genus)) %>%
column_to_rownames("Feature.ID")%>%
#filter(Domain!="Unassigned") %>%
as.matrix()

head(taxa_table_3)


taxa_qza <- read_qza("/Users/stephanie.rosales/Documents/omics_projects//MIR//analysis//seq_run1//marver1/qiime_output//taxa_Marver1_seq1.qza")
taxa_table_1 <- taxa_qza$data %>%
as_tibble() %>% 
#filter(Taxon!="Unassigned") %>%
#select(-Consensus) %>% 
separate(Taxon, sep=";", c("Domain", "Phylum", "Class", "Order", "Family", "Genus", "Species")) %>%
arrange(Feature.ID) %>%
mutate(ASVs =paste('ASV',1:n(), sep = "_")) %>%
mutate(Genus = sub("g__", "", Genus)) %>%
mutate(Genus = sub("_[0-9]+", "", Genus)) %>%
mutate(Species = sub("s__", "", Species)) %>%
mutate(Species = sub("_[0-9]+", "", Species)) %>%
mutate(Family = sub("c__", "", Family)) %>%
mutate(Family = sub("_[0-9]+", "", Family)) %>%
mutate(Family = sub("c__", "", Class)) %>%
mutate(Family = sub("_[0-9]+", "", Class)) %>%
column_to_rownames("Feature.ID")%>%
#filter(Domain!="Unassigned") %>%
as.matrix()

head(taxa_table_1)
dim(taxa_table_1)


taxa_qza <- read_qza("/Users/stephanie.rosales/Documents/omics_projects/MIR/analysis/seq_run1/mifish/qiime_output_sfl_mislabelled_db/taxa_Mifish_seq1_sfl.qza")
taxa_table_mi <- taxa_qza$data %>%
as_tibble() %>% 
#filter(Taxon!="Unassigned") %>%
#select(-Consensus) %>% 
separate(Taxon, sep=";", c("Domain", "Phylum", "Class", "Order", "Family", "Genus", "Species")) %>%
arrange(Feature.ID) %>%
mutate(ASVs =paste('ASV',1:n(), sep = "_")) %>%
column_to_rownames("Feature.ID")%>%
#filter(Domain!="Unassigned") %>%
as.matrix()

head(taxa_table_mi)

Warning message in readLines(file, warn = readLines.warn):
“incomplete final line found on '/var/folders/j4/j37z3nw57wl5490t5t1rhk7h4k4czw/T//RtmpegPLI4/b3310273-c770-404d-8408-fb19d766a6c0/metadata.yaml'”
Warning message:
“Expected 7 pieces. Missing pieces filled with `NA` in 2181 rows [1, 2, 3, 4, 6, 8, 9, 10, 11, 13, 14, 16, 17, 19, 20, 21, 22, 23, 26, 27, ...].”


,Domain,Phylum,Class,Order,Family,Genus,Species,Consensus,ASVs
0003b70454d6e57dfd7fda9b667caeef,NA,Chordata,Actinopteri,NA,NA,NA,NA,1,ASV_1
0013ed0613e0d3a0fd82aac84c315107,NA,Chordata,Actinopteri,Labriformes,Labridae,NA,NA,1,ASV_2
0014f6c3f5307e0d8e1b5451649628b8,NA,Chordata,Actinopteri,NA,NA,NA,NA,1,ASV_3
001ca7ce36a6a1a8062af013da7effb3,NA,Chordata,Actinopteri,Blenniiformes,Tripterygiidae,NA,NA,1,ASV_4
0027bd427ee4d866a7f601fb755a153b,NA,Chordata,Mammalia,Primates,Hominidae,Homo,Homo sapiens,1,ASV_5
002ef3036f081b912ae9adf224485d6b,Unassigned,NA,NA,NA,NA,NA,NA,1,ASV_6


Warning message in readLines(file, warn = readLines.warn):
“incomplete final line found on '/var/folders/j4/j37z3nw57wl5490t5t1rhk7h4k4czw/T//RtmpegPLI4/147c8941-1e23-4ac1-b4c3-c12391fc0133/metadata.yaml'”
Warning message:
“Expected 7 pieces. Missing pieces filled with `NA` in 601 rows [5, 7, 16, 20, 30, 34, 40, 51, 52, 57, 67, 69, 72, 76, 86, 92, 93, 102, 105, 111, ...].”


,Domain,Phylum,Class,Order,Family,Genus,Species,Confidence,ASVs
000f0d9e1c3064a70badeae45f2ba9df,k__Eukaryota_2759,p__Chordata_7711,c__Actinopteri_186623,o__Chaetodontiformes_1545895,c__Actinopteri,Chaetodon,Chaetodon ocellatus,1.0000000,ASV_1
002f0c450b3b355ca242b514b52193b6,k__Eukaryota_2759,p__Chordata_7711,c__Actinopteri_186623,o__Lutjaniformes_2024539,c__Actinopteri,Anisotremus,Anisotremus surinamensis,0.9997477,ASV_2
003e93968a2d5718f84db64839cb6f78,k__Eukaryota_2759,p__Chordata_7711,c__Chondrichthyes_7777,o__Myliobatiformes_117851,c__Chondrichthyes,Aetobatus,Aetobatus narinari,1.0000000,ASV_3
005e51ef90d4797ccf1568b01178f647,k__Eukaryota_2759,p__Chordata_7711,c__Actinopteri_186623,o__Clupeiformes_32446,c__Actinopteri,Opisthonema,Opisthonema oglinum,1.0000000,ASV_4
0070c281ab8ed641c2224cabbf034212,k__Eukaryota_2759,NA,NA,NA,NA,NA,NA,1.0000000,ASV_5
0080f3931a7d1e297e3131d8d5608f36,k__Eukaryota_2759,p__Chordata_7711,c__Actinopteri_186623,o__Centrarchiformes_1489940,c__Actinopteri,Kyphosus,NA,0.9999999,ASV_6


[1] 2685    9

Warning message in readLines(file, warn = readLines.warn):
“incomplete final line found on '/var/folders/j4/j37z3nw57wl5490t5t1rhk7h4k4czw/T//RtmpegPLI4/b5221f12-6970-4f41-b841-e1cb15925c71/metadata.yaml'”
Warning message:
“Expected 7 pieces. Missing pieces filled with `NA` in 1285 rows [1, 5, 23, 24, 33, 42, 43, 45, 48, 49, 50, 54, 62, 63, 69, 75, 87, 89, 95, 109, ...].”


,Domain,Phylum,Class,Order,Family,Genus,Species,Confidence,ASVs
0019b435c1e20c18dedcc08182c39813,Eukaryota,Chordata,Actinopteri,NA,Haemulidae,Haemulon,Haemulon macrostomum,0.9856335,ASV_1
00238158fdd2d6aa0a4d1c493277ac00,Eukaryota,Chordata,Actinopteri,NA,Pomacentridae,Stegastes,Stegastes diencaeus,1.0000000,ASV_2
00260f732981b4453ef2e5777bdb8d08,Eukaryota,Chordata,Actinopteri,Labriformes,Labridae,Scarus,Scarus coeruleus,1.0000000,ASV_3
002697bf26f2728dfdf10de8059bf039,Eukaryota,Chordata,Actinopteri,NA,NA,NA,NA,0.9998789,ASV_4
003c0983d1daf45eedde902221cb8539,Eukaryota,Chordata,Actinopteri,NA,NA,NA,NA,0.8885351,ASV_5
0042ee14d9c27c8afe636c66aeb95d86,Eukaryota,Chordata,Actinopteri,Labriformes,Labridae,Sparisoma,Sparisoma radians,0.9995541,ASV_6


In [58]:
ASV_table_mi_df= as.data.frame(ASV_table_mi)
ASV_table_mi_df <- ASV_table_mi_df[, colSums(ASV_table_mi_df) > 0]

In [59]:

# remove differences in sampleID and sequence ID for MARVER1
setdiff(colnames(ASV_table_1),rownames(meta_m1))
ASV_table_1_df= as.data.frame(ASV_table_1) %>%
select(-MV1_CR, -MV1_HS, -MV1_MSU, -MV1_NTC) 

setdiff(colnames(ASV_table_1_df),rownames(meta_m1))
length(setdiff(colnames(ASV_table_1_df),rownames(meta_m1)))


# remove differences in sampleID and sequence ID for MARVER3
setdiff(colnames(ASV_table_3),rownames(meta_m3))


ASV_table_3_df= as.data.frame(ASV_table_3) %>%
select(-MIRe2_16S_MV3_MSU, -MIRe2_16S_MV3_NTC) 
setdiff(colnames(ASV_table_3_df),rownames(meta_m3))
length(setdiff(colnames(ASV_table_3_df),rownames(meta_m3)))



# remove differences in sampleID and sequence ID for Mifish
setdiff(colnames(ASV_table_mi),rownames(meta_mi))


ASV_table_mi_df= as.data.frame(ASV_table_mi) %>%
select(-MIRe1_12S_MF_16B, -MIRe2_12S_MF_33B, -MIRe2_12S_MF_CR_n_2B, -MIRe3_12S_MF_NTC) 
setdiff(colnames(ASV_table_mi_df),rownames(meta_mi))
length(setdiff(colnames(ASV_table_mi_df),rownames(meta_mi)))



[1] "MV1_CR"  "MV1_HS"  "MV1_MSU" "MV1_NTC"

character(0)

[1] 0

[1] "MIRe2_16S_MV3_MSU" "MIRe2_16S_MV3_NTC"

character(0)

[1] 0

[1] "MIRe1_12S_MF_16B"     "MIRe2_12S_MF_33B"     "MIRe2_12S_MF_CR_n_2B"
[4] "MIRe3_12S_MF_NTC"

character(0)

[1] 0

In [60]:
setdiff(row.names(meta_mi),colnames(ASV_table_mi_df))
length(setdiff(colnames(ASV_table_mi_df),rownames(meta_mi)))

character(0)

[1] 0

### Prepare metadata for SCRUB

In [61]:
scrub_meta_3 <- 
 #data.frame(sample_data(ps_3))  %>% as.data.frame() %>%
meta_m3 %>%
  transmute(
    is_control   = samp_category != "sample",
    sample_type  = samp_category,
    #sample_well  = extract_number
    #sample_type  = neg_cont_type
  ) %>%
  as.data.frame()

# Preserve original rownames from meta
rownames(scrub_meta_3) <- rownames(meta_m3)

# Quick QC
scrub_meta_3 %>% count(is_control, sample_type)
head(scrub_meta_3)
dim(scrub_meta_3)

# Reorder metadata to match ASV table
scrub_meta_3 <- scrub_meta_3[rownames(t(ASV_table_3_df)), ]

is_control,sample_type,n
<lgl>,<chr>,<int>
FALSE,sample,87
TRUE,negative control,8


,is_control,sample_type
,<lgl>,<chr>
MIRe1_16S_MV3_10A,FALSE,sample
MIRe1_16S_MV3_10B,FALSE,sample
MIRe1_16S_MV3_11A,FALSE,sample
MIRe1_16S_MV3_11B,FALSE,sample
MIRe1_16S_MV3_12A,FALSE,sample
MIRe1_16S_MV3_12B,FALSE,sample


[1] 95  2

In [62]:
scrub_meta_1 <- 
 #data.frame(sample_data(ps_3))  %>% as.data.frame() %>%
meta_m1 %>%
  transmute(
    is_control   = samp_category != "sample",
    sample_type  = samp_category,
    #sample_well  = extract_number,
    #sample_type  = neg_cont_type
  ) %>%
  as.data.frame()

# Preserve original rownames from meta
rownames(scrub_meta_1) <- rownames(meta_m1)

# Quick QC
scrub_meta_1 %>% count(is_control, sample_type)
head(scrub_meta_1)
dim(scrub_meta_1)

# Reorder metadata to match ASV table
scrub_meta_1 <- scrub_meta_1[rownames(t(ASV_table_1_df)), ]

is_control,sample_type,n
<lgl>,<chr>,<int>
FALSE,sample,87
TRUE,negative control,7


,is_control,sample_type
,<lgl>,<chr>
MV1_10A,FALSE,sample
MV1_10B,FALSE,sample
MV1_11A,FALSE,sample
MV1_11B,FALSE,sample
MV1_12A,FALSE,sample
MV1_12B,FALSE,sample


[1] 94  2

In [63]:
scrub_meta_mi <- 
 #data.frame(sample_data(ps_3))  %>% as.data.frame() %>%
meta_mi %>%
  transmute(
    is_control   = samp_category != "sample",
    sample_type  = samp_category,
    #sample_well  = extract_number,
    #   sample_type  = neg_cont_type
  ) %>%
  as.data.frame()

# Preserve original rownames from meta
rownames(scrub_meta_mi) <- rownames(meta_mi)

# Quick QC
scrub_meta_mi %>% count(is_control, sample_type)
head(scrub_meta_mi)
dim(scrub_meta_mi)

# Reorder metadata to match ASV table
scrub_meta_mi <- scrub_meta_mi[rownames(t(ASV_table_mi_df)), ]

is_control,sample_type,n
<lgl>,<chr>,<int>
FALSE,sample,142
TRUE,negative control,19


,is_control,sample_type
,<lgl>,<chr>
MIRe1_12S_MF_10A,FALSE,sample
MIRe1_12S_MF_10B,FALSE,sample
MIRe1_12S_MF_11A,FALSE,sample
MIRe1_12S_MF_11B,FALSE,sample
MIRe1_12S_MF_12A,FALSE,sample
MIRe1_12S_MF_12B,FALSE,sample


[1] 161   2

### Prepare SCRUB sequence count table

In [64]:
ps_3 = phyloseq(otu_table(ASV_table_3_df, taxa_are_rows=TRUE), 
               sample_data(meta_m3),
tax_table(taxa_table_3))
ps_3  

ps_1 = phyloseq(otu_table(ASV_table_1_df, taxa_are_rows=TRUE), 
               sample_data(meta_m1),
tax_table(taxa_table_1))
ps_1 %>% tax_fix()


ps_mi = phyloseq(otu_table(ASV_table_mi_df, taxa_are_rows=TRUE), 
               sample_data(meta_mi),
tax_table(taxa_table_mi))
ps_mi %>% tax_fix()

ps_mi <- prune_samples(sample_sums(ps_mi) > 0, ps_mi)
ps_mi

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 3646 taxa and 94 samples ]
sample_data() Sample Data:       [ 94 samples by 64 sample variables ]
tax_table()   Taxonomy Table:    [ 3646 taxa by 9 taxonomic ranks ]

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 2685 taxa and 92 samples ]
sample_data() Sample Data:       [ 92 samples by 64 sample variables ]
tax_table()   Taxonomy Table:    [ 2685 taxa by 9 taxonomic ranks ]

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 4712 taxa and 161 samples ]
sample_data() Sample Data:       [ 161 samples by 175 sample variables ]
tax_table()   Taxonomy Table:    [ 4712 taxa by 9 taxonomic ranks ]

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 4712 taxa and 159 samples ]
sample_data() Sample Data:       [ 159 samples by 175 sample variables ]
tax_table()   Taxonomy Table:    [ 4712 taxa by 9 taxonomic ranks ]

In [65]:
otu_df_3 <- as.data.frame(t(otu_table(ps_3)))
otu_df_1 <- as.data.frame(t(otu_table(ps_1)))
otu_df_mi <- as(otu_table(t(ps_mi)), "matrix")


In [72]:
dim(otu_df_mi)
dim(scrub_meta_mi)

dim(otu_df_3)
dim(scrub_meta_3)

[1]  159 4712

[1] 159   2

[1]   94 3646

[1] 94  2

In [70]:
setdiff(rownames(scrub_meta_mi),rownames(otu_df_mi))
length(setdiff(colnames(scrub_meta_mi),rownames(otu_df_mi)))

setdiff(row.names(otu_df_mi),row.names(scrub_meta_mi))
length(setdiff(row.names(otu_df_mi),row.names(scrub_meta_mi)))

[1] "MIRe2_12S_MF_CFS_n_2A"     "MIRe2_12S_MF_ExBlank_p2ws"

[1] 2

character(0)

[1] 0

In [71]:
rows_to_remove <- c("MIRe2_12S_MF_CFS_n_2A", "MIRe2_12S_MF_ExBlank_p2ws")
scrub_meta_mi <- scrub_meta_mi[!rownames(scrub_meta_mi) %in% rows_to_remove, ]
setdiff(rownames(scrub_meta_mi),rownames(otu_df_mi))
length(setdiff(colnames(scrub_meta_mi),rownames(otu_df_mi)))

character(0)

[1] 2

### Run SCRUB 

In [73]:
scr_out_3 <- SCRuB(otu_df_3,
                 scrub_meta_3,
                   #control_order =  c('field negative', 'extraction negative')
                  )

[1] "Did not find well metadata, running SCRuB without the spatial component"
[1] "SCRuBbing away contamination in the negative control controls..."


In [74]:
scr_out_1 <- SCRuB(otu_df_1,
                 scrub_meta_1,
                   #control_order =  c('field negative', 'extraction negative')
                  )

[1] "Did not find well metadata, running SCRuB without the spatial component"
[1] "SCRuBbing away contamination in the negative control controls..."


In [75]:
scr_out_mi <- SCRuB(otu_df_mi,
                 scrub_meta_mi, 
                   #control_order =  c('field negative', 'extraction negative')
                  )

[1] "Did not find well metadata, running SCRuB without the spatial component"
[1] "SCRuBbing away contamination in the negative control controls..."


In [76]:
decontaminated_m3 <- as.data.frame(scr_out_3$decontaminated_samples)
head(decontaminated_m3, n=2)
dim(decontaminated_m3)

decontaminated_m1 <- as.data.frame(scr_out_1$decontaminated_samples)
head(decontaminated_m1, n=2)
dim(decontaminated_m1)


decontaminated_mi <- as.data.frame(scr_out_mi$decontaminated_samples)
head(decontaminated_mi, n=2)
dim(decontaminated_mi)

,24603dde52fe88f5148c3c60c5d6e1b2,ece06f5929d636c6850f5444574486f5,52101314bc187461c2d4c5b5dabe6a89,33534b6a6f23bd8acb02bf8490fd822f,07b1d9f7e7dce81c3b240e3e3f5f997b,a23cbfdeaeb29f2bbeefad65c4c4b2f8,cfd8220f906a1a19cbc73ef31b76021a,d4878bc3fe3c131cc7ecbf55edfb7425,88f1a1d09d11fe06660d80c1ba4d0602,bb3462b2d489dce1d47ac04d740ee4b0,⋯,25377f38430f0105aa6864ab9fcc91f0,3e62d96191146acaa8f2d2b78ce1b517,3994488d4d6cbc2f6404f133da00fc1a,749e8768124864616f3365f02dce7989,204420ca4c8d6408a9b4b9cbf9b2b594,991e059c0c201fac5d606103618d1420,33bf4041a5f273a89407b355992ff65a,f4289d9ffb552441dd6742d731a328a9,6a2d26dc3c5a733b337d15ca60d246a7,f157408f0b2f33dc25486a3dd7a08464
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
MIRe1_16S_MV3_10A,0,0,0,0,8247,27,0,0,39894,28104,⋯,0,0,0,0,0,0,0,0,0,0
MIRe1_16S_MV3_10B,0,0,0,6988,8273,11450,211,0,15000,112,⋯,0,0,0,0,0,0,0,0,0,0
MIRe1_16S_MV3_11A,0,3688,0,0,0,351,0,0,2431,932,⋯,0,0,0,0,0,0,0,0,0,0
MIRe1_16S_MV3_11B,0,0,0,12758,2272,0,16928,0,0,64523,⋯,0,0,0,0,0,0,0,0,0,0
MIRe1_16S_MV3_12A,0,15911,54595,8186,25363,0,2282,0,0,12877,⋯,0,0,0,0,0,0,0,0,0,0
MIRe1_16S_MV3_12B,0,0,0,0,0,0,138,0,0,449,⋯,0,0,0,0,0,0,0,0,0,0


[1]   87 3646

,e03b6ee0df42c3ca44ea26ed6c4d27aa,7da05e198a2c205b708d4f3ebeb5ee12,83c356d0ebc3a8e44e76aab2fa442c6d,ef4d7a661ddc4fbcce5cef9569d8c574,c88db9b123be52c2969a0aaf38ee3128,68fbbdf8bd741af2251a9a1c5f4a5162,23a65576114e6d5cffd94635c5bb6e60,9978fc802c7bf691234eef89494b0ec4,7849d39c640cecb0070c4aa6eb65657c,59397773cc737943383d87a7db391c6a,⋯,6f9712e56079f0d6002b2c1d053ef609,3218aaa7642b62a54e96fe488cd8f4d4,aff7cf8946e2b926e456f3f95e598cf7,b08946dd4c4340e48f661b0a4029a1c0,6f5aa87f4b93c070974f8d0cbf63f22c,8011dd36c1c8ecb9262af40146487757,c456f5262932934ab15e6d5ca6f0ac66,a0ed7ce6c9fd368db9f59b73c259f088,e1f72b86420b0ab29223b3159e95d3eb,afeb62469250fd6d4d40289b29a06c40
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
MV1_10A,0,0,0,93,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
MV1_10B,0,0,0,64238,0,0,4663,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
MV1_11A,0,0,0,0,0,0,6859,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
MV1_11B,0,0,0,0,0,4955,19052,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
MV1_12A,0,0,0,0,0,0,59977,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
MV1_12B,0,0,0,0,0,235,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


[1]   87 2685

,5e704746baef0d3e632a7def74b449b9,addd57bd3b3937b61b32dece26f8f166,918608adb7967723526fc0c0b9026869,c75541bd3408a7ac9fb378dd25f9d228,37c2d350be5db1d723b3d091cfcaa492,6d2fd6c7c4e302d4fd495d55d141a568,20f92d11f4596710223e0ea1622b8151,b788ed4205861dfbe732bceee5d222aa,2c4c4cb938987a6880e8975198de429b,a02ef7f5fb571ba596f7846802d3ce4b,⋯,257fbb1e3b68eb28e3e336e5d92ff42d,79b6b74d0ad166da5d59209a59e24835,ce15bca5d810d710f790fdc1cbc3ccc7,bacc275a7008dc82d5ba591cfca4d1ad,9db82c74ea2a0234fc5542d6675196ae,f4a703890cd020bed7714325620025c7,155ca14a2163c210abb09526df5cf797,8c554b4b66ccbd21d997258b335bc501,6033da90531df3daef6c2cedb7a6cdce,cca419ad33609d18d00974fd3bf9b235
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
MIRe1_12S_MF_10A,0,25957,0,25496,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
MIRe1_12S_MF_10B,42487,0,0,4065,0,0,0,3576,5515,0,⋯,0,0,0,0,0,0,0,0,0,0
MIRe1_12S_MF_11A,0,0,0,2925,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
MIRe1_12S_MF_11B,0,31805,13841,43118,0,0,11436,0,0,4554,⋯,0,0,0,0,0,0,0,0,0,0
MIRe1_12S_MF_12A,0,16427,12576,2776,0,0,0,2254,0,462,⋯,0,0,0,0,0,0,0,0,0,0
MIRe1_12S_MF_12B,0,0,1130,0,1185,0,4279,3668,0,0,⋯,0,0,0,0,0,0,0,0,0,0


[1]  142 4712

In [84]:
write.csv(decontaminated_m3, "/Users/stephanie.rosales/Documents/omics_projects/MIR/Fish_comparison/Working_files/ASV_table_MV3_scrub.csv")

## Identify species that may be missclassified

### Filter taxa SFL and GOM data

In [82]:
gomexSFL= read.csv("/Users/stephanie.rosales/Documents/omics_projects/MIR/taxon_lists/gomex_and_SFLnames.csv",
             header = TRUE) %>%
rename(c("scientificName"="ScientificName_accepted")) %>%
select(scientificName)  #%>%
#mutate_if(is.character, na_if, "")

tail(gomexSFL)
dim(gomexSFL)

obis= read.csv("/Users/stephanie.rosales/Documents/omics_projects/MIR/taxon_lists/gomex_obis_allfish_names.tsv",
             header = TRUE, sep="\t") %>%
select(scientificName)  %>%
mutate_if(is.character, na_if, "")
tail(obis)
dim(obis)

,scientificName
,<chr>
993,
994,
995,
996,
997,
998,


[1] 998   1

,scientificName
,<chr>
1513,Mola mola
1514,Pontinus nematophthalmus
1515,Grammicolepis brachiusculus
1516,Acromycter atlanticus
1517,Symphurus stigmosus
1518,Xenomystax bidentatus


[1] 1518    1

In [83]:
#merge both lists and then remove duplicates
merged_fish_names = full_join(obis, gomexSFL) %>%
distinct(scientificName)
tail(merged_fish_names)
dim(merged_fish_names)


Joining with `by = join_by(scientificName)`


,scientificName
,<chr>
1679,Centrophorus squamosus
1680,Centrophorus tessellatus
1681,Deania calceus
1682,Centroscyllium fabricii
1683,Etmopterus carteri
1684,Etmopterus pusillus


[1] 1684    1

In [29]:
#Find differences between curated (Gomex) and assigned taxa for MARVER1
setdiff(
  unique(as.data.frame(tax_table(taxa_table_1))$Species),
  unique(merged_fish_names$scientificName)
)
length(setdiff(
  unique(as.data.frame(tax_table(taxa_table_1))$Species),
  unique(merged_fish_names$scientificName)
))


# Taxa that are not in curated (Gomex) but in taxa files

taxa_not_in_merged_1 <- as.data.frame(tax_table(taxa_table_1)) %>%
filter(Confidence>=0.99) %>%
  mutate(Species = trimws(tolower(Species))) %>% #Converts all characters to lowercase
  rename(scientificName = Species) %>%
  anti_join(
    merged_fish_names %>%
      mutate(scientificName = trimws(tolower(scientificName))), #Converts all characters to lowercase
    by = "scientificName"
  ) %>%
  filter(Class != "Mammalia",
         Phylum != "Mollusca",
         Genus != "Bos",
         Genus != "Canis",
         Genus != "Gallus",
          Genus != "Homo",
         Genus != "Pan",
         Genus != "Sus",
         Genus!="Meleagris",
         Phylum != "Gallus",
         !is.na(scientificName)) %>%
  arrange(scientificName) %>%
select("Phylum", "Class", "Order","Family","Genus","scientificName")  %>%
filter(Class!="Actinopteri")  %>%
#select("scientificName") %>%
distinct()

taxa_not_in_merged_1

unique(taxa_not_in_merged_1$scientificName)
length(unique(taxa_not_in_merged_1$scientificName))

[1] NA                            "Homo sapiens"               
 [3] "Homo heidelbergensis"        "Hemiramphus far"            
 [5] "Gallus gallus"               "Platybelone argala"         
 [7] "Sus scrofa"                  "Pan paniscus"               
 [9] "Cinachyrella kuekenthali"    "Carangoides bartholomaei"   
[11] "Opistognathus jacksoniensis" "Bos primigenius"            
[13] "Bos taurus"                  "Pempheris adusta"           
[15] "Gobiidae sp."                "Chromis cyanea"             
[17] "Canis lupus"                 "Mulloidichthys vanicolensis"
[19] "Brachygenys chrysargyrea"    "Abudefduf sparoides"        
[21] "Procyon lotor"               "Oncorhynchus gorbuscha"     
[23] "Chelonia mydas"              "Notemigonus crysoleucas"    
[25] "Sus cebifrons"               "Xyrias revulsus"            
[27] "Bos indicus"                 "Culcita novaeguineae"       
[29] "Sparisoma axillare"          "Pterois volitans"           
[31] "Pseudogramma paucilepis"     "Cheilopogon doederleinii"   
[33] "Antennarius biocellatus"     "Neoniphon aurolineatus"     
[35] "Thalassoma jansenii"         "Meleagris gallopavo"        
[37] "Rhinesomus triqueter"

[1] 37

,Phylum,Class,Order,Family,Genus,scientificName
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
5cfeed9ca7f9c7ce05701af0ab614f2d,p__Chordata_7711,c__Actinopteri_186623,o__Lutjaniformes_2024539,c__Actinopteri,Brachygenys,brachygenys chrysargyrea
1c58162161dc436d0ca988d73d003c21,p__Chordata_7711,c__Actinopteri_186623,o__Carangiformes_1489907,c__Actinopteri,Carangoides,carangoides bartholomaei
648bc8c5823fb367580f565491247406,p__Chordata_7711,c__class_Testudines_8459,o__Testudines_8459,c__class_Testudines,Chelonia,chelonia mydas
31dbb4e637b8ad9209747c9d80b72bf4,p__Chordata_7711,c__Actinopteri_186623,o__order_Pomacentridae_30863,c__Actinopteri,Chromis,chromis cyanea
169a14f60d555bea135059e05910e48f,p__Porifera_6040,c__Demospongiae_6042,o__Tetractinellida_1779164,c__Demospongiae,Cinachyrella,cinachyrella kuekenthali
2d213d2afe422c3e50186542190820af,p__Chordata_7711,c__Actinopteri_186623,o__Gobiiformes_1489878,c__Actinopteri,genus_Gobiidae sp.,gobiidae sp.
ed76db8e52546ffe5487e66f1e47ad81,p__Chordata_7711,c__Actinopteri_186623,o__Holocentriformes_1490028,c__Actinopteri,Neoniphon,neoniphon aurolineatus
66cea1d286a7f54fd50dd7bfda152c5a,p__Chordata_7711,c__Actinopteri_186623,o__Cypriniformes_7952,c__Actinopteri,Notemigonus,notemigonus crysoleucas
2a2f2caefb8d83b7a928bb72a1135445,p__Chordata_7711,c__Actinopteri_186623,o__Acropomatiformes_3051910,c__Actinopteri,Pempheris,pempheris adusta


[1] "brachygenys chrysargyrea" "carangoides bartholomaei"
 [3] "chelonia mydas"           "chromis cyanea"          
 [5] "cinachyrella kuekenthali" "gobiidae sp."            
 [7] "neoniphon aurolineatus"   "notemigonus crysoleucas" 
 [9] "pempheris adusta"         "platybelone argala"      
[11] "procyon lotor"            "pseudogramma paucilepis" 
[13] "pterois volitans"         "rhinesomus triqueter"    
[15] "xyrias revulsus"

[1] 15

| Taxon                    | Common Name        | Primary Region / Distribution | Plausible in Caribbean eDNA? | Notes                             |
| ------------------------ | ------------------ | ----------------------------- | ---------------------------- | --------------------------------- |
| Carangoides bartholomaei | Yellow jack        | Western Atlantic / Caribbean  | Yes                          | Common pelagic reef fish          |
| Chelonia mydas           | Green sea turtle   | Global tropical oceans        | Yes                          | Common in Florida waters          |
| Chromis cyanea           | Blue chromis       | Caribbean                     | Yes                          | Common reef fish                  |
| Neoniphon aurolineatus   | Squirrelfish       | Western Atlantic              | Yes                          | Caribbean reef fish               |
| Pseudogramma paucilepis  | Reef basslet       | Western Atlantic              | Yes                          | Caribbean reef species            |
| Pterois volitans         | Lionfish           | Indo-Pacific (invasive)       | Yes                          | Established invasive in Caribbean |
| Rhinesomus triqueter     | Reef squirrelfish  | Caribbean                     | Yes                          | Reef fish                         |
| Procyon lotor            | Raccoon            | North America                 | Yes                          | Likely terrestrial runoff DNA     |
| Gobiidae                 | Gobies             | Global                        | Yes                          | Family-level ID                   |
| Cinachyrella kuekenthali | Marine sponge      | Indo-Pacific mostly           | Possibly                     | Sponge taxonomy uncertain         |
| Platybelone argalus      | Needlefish         | Tropical oceans               | Possibly                     | Widespread genus                  |
| Brachygenys chrysargyrea | Uncertain taxonomy | Unclear                       | Uncertain                    | Taxonomic ambiguity               |
| Pempheris adusta         | Sweeper fish       | Indo-Pacific                  | No                           | Likely taxonomic misassignment    |
| Notemigonus crysoleucas  | Freshwater fish    | North America freshwater      | No                           | Freshwater species                |
| Xyrias revulsus          | Snake eel          | Indo-Pacific                  | No                           | Regional mismatch                 |


Pempheris adusta = no regional similarties; 
could be
Xyrias revulsus = Ophisurus macrorhynchos

In [30]:
#Find differences between curated (Gomex) and assigned taxa for MARVER3
setdiff(
  unique(as.data.frame(tax_table(taxa_table_3))$Species),
  unique(merged_fish_names$scientificName)
)
length(setdiff(
  unique(as.data.frame(tax_table(taxa_table_3))$Species),
  unique(merged_fish_names$scientificName)
))


# Taxa that are not in curated (Gomex) but in taxa files

taxa_not_in_merged_3 <- as.data.frame(tax_table(taxa_table_3)) %>%
  mutate(Species = trimws(tolower(Species))) %>% #Converts all characters to lowercase
  rename(scientificName = Species) %>%
  anti_join(
    merged_fish_names %>%
      mutate(scientificName = trimws(tolower(scientificName))), #Converts all characters to lowercase
    by = "scientificName"
  ) %>%
  filter(Class != "Mammalia",
         Phylum != "Mollusca",
         Genus != "Bos",
         Genus != "Canis",
         Genus != "Gallus",
          Genus != "Homo",
         Genus != "Pan",
         Genus != "Sus",
         Genus!="Meleagris",
         Phylum != "Gallus",
         !is.na(scientificName)) %>%
  arrange(scientificName) %>%
select("Phylum", "Class", "Order","Family","Genus","scientificName")  %>%
filter(Class!="Actinopteri")  %>%
#select("scientificName") %>%
distinct()

taxa_not_in_merged_3

unique(taxa_not_in_merged_3$scientificName)
length(unique(taxa_not_in_merged_3$scientificName))

[1] NA                             "Homo sapiens"                
 [3] "Spondylus ambiguus"           "NA"                          
 [5] "Coryphopterus urospilus"      "Carangoides bartholomaei"    
 [7] "Sphyraena jello"              "Ophichthus erabo"            
 [9] "Cyclothone atraria"           "Tigrigobius gemmatus"        
[11] "Aglaura hemistoma"            "Trachinotus falcatus"        
[13] "Dysidea avara"                "Platybelone argala"          
[15] "Echeneis naucratoides"        "Eucinostomus currani"        
[17] "Gillellus semicinctus"        "Carangoides ruber"           
[19] "Lythrypnus sp. AC-2023"       "Gallus gallus"               
[21] "Ogilbichthys sp. USNM 349073" "Canis lupus"                 
[23] "Platanista minor"             "Myrichthys maculosus"        
[25] "Bos taurus"                   "Neoniphon opercularis"       
[27] "Sargocentron punctatissimum"  "Alloclinus holderi"

[1] 28

,Phylum,Class,Order,Family,Genus,scientificName
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1797e67d1bf2cbd03e3f3b67184f7b16,Cnidaria,Hydrozoa,Trachymedusae,Rhopalonematidae,Aglaura,aglaura hemistoma
2094f16b49540a52de156e3c11ef1dab,Porifera,Demospongiae,Dictyoceratida,Dysideidae,Dysidea,dysidea avara
05dbcf4106fa6e047c3d2468cb29b278,NA,NA,NA,NA,NA,na
0ed2a119534e3e3a7bd3bb237f0b032c,Cnidaria,Anthozoa,Scleractinia,NA,NA,na
367d5d6ab8c51c02032c7f618f665210,Porifera,Demospongiae,Haplosclerida,Callyspongiidae,Callyspongia,na
8fd9ec1b22d5c38a48976482d8559b9e,Chordata,NA,NA,NA,NA,na


[1] "aglaura hemistoma" "dysidea avara"     "na"

[1] 3